# Housing Price Prediction Using Multiple Linear Regression

#### <center> ***Domain: Real Estate***

---

<center><img src="https://raw.githubusercontent.com/Masterx-AI/Project_Housing_Price_Prediction_/main/hs.jpg" style="width: 700px;"/>

---

# <center> Strategic Plan of Action:

**We aim to solve the problem statement by creating a plan of action, Here are some of the necessary steps:**
1. Data Exploration
2. Exploratory Data Analysis (EDA)
3. Data Pre-processing
4. Data Manipulation
5. Feature Selection/Extraction
6. Multiple Linear Regression Modelling
7. Project Outcomes & Conclusion

---

# <center>1. Data Exploration

In [ ]:
#Importing the basic libraries

import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

from sklearn.feature_selection import RFE
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from statsmodels.stats.outliers_influence import variance_inflation_factor

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = [10,6]

import warnings 
warnings.filterwarnings('ignore')

In [ ]:
from pathlib import Path

FIGURES_DIR = Path("../outputs/figures")
TABLES_DIR = Path("../outputs/tables")

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

print("Figures directory:", FIGURES_DIR.resolve())
print("Tables directory:", TABLES_DIR.resolve())

In [ ]:
#Importing the dataset

df = pd.read_csv("../data/raw/Housing.csv")

display(df.head())

target = 'price'
features = [i for i in df.columns if i not in [target]]

original_df = df.copy(deep=True)

print('\n\033[1mInference:\033[0m The Dataset consists of {} features & {} samples.'.format(df.shape[1], df.shape[0]))

In [ ]:
#Checking the dtypes of all the columns

df.info()

In [ ]:
#Checking number of unique rows in each feature

df.nunique().sort_values()

In [ ]:
#Checking number of unique rows in each feature

nu = df[features].nunique().sort_values()
nf = []; cf = []; nnf = 0; ncf = 0; #numerical & categorical features

for i in range(df[features].shape[1]):
    if nu.values[i]<=16:cf.append(nu.index[i])
    else: nf.append(nu.index[i])

print('\n\033[1mInference:\033[0m The Dataset has {} numerical & {} categorical features.'.format(len(nf),len(cf)))

In [ ]:
#Checking the stats of all the columns

display(df.describe())

---

# <center> 2. Exploratory Data Analysis (EDA)

In [ ]:
plt.figure(figsize=(8, 4))

sns.histplot(
    data=df,
    x=target,
    bins=30,
    kde=True
)

plt.title("Distribution of Housing Prices")
plt.xlabel("Price")
plt.ylabel("Frequency")
plt.tight_layout()

plt.savefig(
    FIGURES_DIR / "01_price_distribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

**Inference:** The target variable is right-skewed, with most houses concentrated in the lower-to-middle price range and several high-price observations forming a long right tail.

In [ ]:
print('\033[1m' + ' ' * 33 + 'Visualising Categorical Features:' + ' ' * 33)

n = len(cf)
ncols = 3
nrows = int(np.ceil(n / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(20, 5 * nrows))

# Flatten axes để dễ truy cập theo index tuyến tính
if nrows == 1 and ncols == 1:
    axes = np.array([[axes]])
elif nrows == 1:
    axes = axes.reshape(1, -1)
elif ncols == 1:
    axes = axes.reshape(-1, 1)

for i, col in enumerate(cf):
    ax = axes[i // ncols, i % ncols]
    temp_df = pd.DataFrame(df.groupby([col])[target].mean()).reset_index()
    ax.bar(temp_df[col].astype(str), temp_df[target])
    ax.set_title(col)
    ax.set_xlabel(col)
    ax.set_ylabel('Mean Price')
    ax.tick_params(axis='x', rotation=45)

# Ẩn các ô subplot dư thừa
for j in range(i + 1, nrows * ncols):
    axes[j // ncols, j % ncols].set_visible(False)

plt.tight_layout()

plt.savefig(
    FIGURES_DIR / "02_categorical_features_mean_price.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


In [ ]:
print('\033[1m' + ' ' * 33 + 'Visualising Numerical Features:' + ' ' * 33)

fig, axes = plt.subplots(1, len(nf), figsize=(15, 5))
if len(nf) == 1:
    axes = [axes]
for i, col in enumerate(nf):
    axes[i].scatter(df[col], df[target], alpha=0.5)
    axes[i].set_title(col)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Price')
plt.tight_layout()

plt.savefig(
    FIGURES_DIR / "03_area_vs_price.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

---

# <center> 3. Data Pre-processing

## 3a. Missing Value Analysis

In [ ]:
#Checking for missing values

print('\033[1mNull Values Count\033[0m')
print(df.isnull().sum())
print('\n\033[1mInference:\033[0m The Dataset has no missing values.')

## 3b. Duplicate Value Analysis

In [ ]:
#Checking for duplicate rows

print('Number of duplicate rows:', df.duplicated().sum())
print('\n\033[1mInference:\033[0m The Dataset has no duplicate rows.')

## 3c. Outlier Analysis

In [ ]:
#Visualizing outliers using boxplots

fig, axes = plt.subplots(1, len(nf), figsize=(15, 5))
if len(nf) == 1:
    axes = [axes]
for i, col in enumerate(nf):
    axes[i].boxplot(df[col])
    axes[i].set_title(col)
plt.suptitle("Boxplots - Numerical Features", fontsize=14)
plt.tight_layout()

plt.savefig(
    FIGURES_DIR / "04_numerical_features_boxplots.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
#Removing outliers using IQR method

df1 = df.copy()
df3 = df.copy()

for i in df1[nf].columns:
    Q1 = df1[i].quantile(0.25)
    Q3 = df1[i].quantile(0.75)
    IQR = Q3 - Q1
    df1 = df1[df1[i] <= (Q3+(1.5*IQR))]
    df1 = df1[df1[i] >= (Q1-(1.5*IQR))]
    df1 = df1.reset_index(drop=True)
display(df1.head())
print('\n\033[1mInference:\033[0m\nBefore removal of outliers, The dataset had {} samples.'.format(df3.shape[0]))
print('After removal of outliers, The dataset now has {} samples.'.format(df1.shape[0]))

---

# <center> 4. Data Manipulation

In [ ]:
#Splitting the data into training & testing sets

df = df1.copy()
df.columns=[i.replace('-','_') for i in df.columns]

m=[]
for i in df.columns.values:
    m.append(i.replace(' ','_'))
    
df.columns = m
X = df.drop([target],axis=1)
Y = df[target]
Train_X, Test_X, Train_Y, Test_Y = train_test_split(X, Y, train_size=0.8, test_size=0.2, random_state=100)
Train_X.reset_index(drop=True,inplace=True)

print('Original set  --> ',X.shape,Y.shape,'\nTraining set  --> ',Train_X.shape,Train_Y.shape,'\nTesting set   --> ', Test_X.shape,'', Test_Y.shape)

In [ ]:
#Standardization on Training set

print('\033[1m' + ' ' * 41 + 'Standardization on Training set' + ' ' * 41)

# Encoding categorical variables
Train_X = pd.get_dummies(Train_X, drop_first=True)
Test_X = pd.get_dummies(Test_X, drop_first=True)

# Align test set columns with training set
Test_X = Test_X.reindex(columns=Train_X.columns, fill_value=0)

scaler = StandardScaler()
Train_X_std = pd.DataFrame(scaler.fit_transform(Train_X), columns=Train_X.columns)
Test_X_std = pd.DataFrame(scaler.transform(Test_X), columns=Test_X.columns)

display(Train_X_std.describe())

---

# <center> 5. Feature Selection

## 5a. Manual Method - VIF (Variance Inflation Factor)

VIF is used to detect multicollinearity among the independent variables. A high VIF value (typically > 10) indicates strong multicollinearity, which may distort the regression coefficients.

In [ ]:
#Checking for Multicollinearity using VIF

vif_data = pd.DataFrame()
vif_data["Feature"] = Train_X_std.columns
vif_data["VIF"] = [variance_inflation_factor(Train_X_std.values, i) for i in range(Train_X_std.shape[1])]
vif_data = vif_data.sort_values('VIF', ascending=False).reset_index(drop=True)
print('\n\033[1mVIF Analysis:\033[0m')
display(vif_data)

## 5b. Automatic Method - RFE (Recursive Feature Elimination)

RFE is used to select the most relevant features for the Multiple Linear Regression model by recursively removing the least important features.

In [ ]:
LR = LinearRegression()
rfe = RFE(LR, n_features_to_select=10)
rfe.fit(Train_X_std, Train_Y)

rfe_df = pd.DataFrame({'Feature': Train_X_std.columns, 'Selected': rfe.support_, 'Ranking': rfe.ranking_})
rfe_df = rfe_df.sort_values('Ranking').reset_index(drop=True)
print('\n\033[1mRFE Selected Features:\033[0m')
display(rfe_df)

print('\nSelected features:', list(Train_X_std.loc[:,rfe.support_].columns))

# ✅ Dùng rfe.estimator_ thay vì LR
fitted_LR = rfe.estimator_
print(np.sqrt(mean_squared_error(Train_Y, fitted_LR.predict(Train_X_std.loc[:,rfe.support_]))))
print(np.sqrt(mean_squared_error(Test_Y,  fitted_LR.predict(Test_X_std.loc[:,rfe.support_]))))


---

# <center> 6. Predictive Modelling with Multiple Linear Regression

## Objective

Build and evaluate a Multiple Linear Regression model for predicting housing prices using the processed feature set.

In [ ]:
#Let us first define a function to evaluate our model

#Let us first define a function to evaluate our model

model_evaluation = pd.DataFrame(
    index=["Multiple Linear Regression"],
    columns=[
        "Train-R2",
        "Test-R2",
        "Train-RSS",
        "Test-RSS",
        "Train-MSE",
        "Test-MSE",
        "Train-RMSE",
        "Test-RMSE"
    ]
)

rc = np.random.choice(
    Train_X_std.loc[:, Train_X_std.nunique() >= 50].columns.values,
    1,
    replace=False
) if len(Train_X_std.loc[:, Train_X_std.nunique() >= 50].columns) > 0 else Train_X_std.columns[:1]

def evaluate_mlr(train_predictions, test_predictions):
    plt.figure(figsize=(10, 6))

    for i in rc:
        plt.scatter(
            Train_X_std[i],
            Train_Y,
            label='Actual',
            alpha=0.6
        )
        plt.scatter(
            Train_X_std[i],
            train_predictions,
            label='Prediction',
            alpha=0.6
        )
        plt.title(f'Actual vs Predicted Prices by {i}')
        plt.xlabel(i)
        plt.ylabel('Price')
        plt.legend()
        break

    plt.tight_layout()
    plt.savefig(
        FIGURES_DIR / "05_actual_vs_predicted_by_feature.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()

    #Evaluating the Multiple Linear Regression Model

    print('\n\n{}Training Set Metrics{}'.format('-'*20, '-'*20))
    print('\nR2-Score on Training set --->',round(r2_score(Train_Y, train_predictions),4))
    print('Residual Sum of Squares (RSS) on Training set  --->',round(np.sum(np.square(Train_Y-train_predictions)),4))
    print('Mean Squared Error (MSE) on Training set       --->',round(mean_squared_error(Train_Y, train_predictions),4))
    print('Root Mean Squared Error (RMSE) on Training set --->',round(np.sqrt(mean_squared_error(Train_Y, train_predictions)),4))

    print('\n{}Testing Set Metrics{}'.format('-'*20, '-'*20))
    print('\nR2-Score on Testing set --->',round(r2_score(Test_Y, test_predictions),4))
    print('Residual Sum of Squares (RSS) on Testing set  --->',round(np.sum(np.square(Test_Y-test_predictions)),4))
    print('Mean Squared Error (MSE) on Testing set       --->',round(mean_squared_error(Test_Y, test_predictions),4))
    print('Root Mean Squared Error (RMSE) on Testing set --->',round(np.sqrt(mean_squared_error(Test_Y, test_predictions)),4))
    print('\n{}Residual Plots{}'.format('-'*20, '-'*20))
    
    model_evaluation.loc['Multiple Linear Regression','Train-R2']  = round(r2_score(Train_Y, train_predictions),4)
    model_evaluation.loc['Multiple Linear Regression','Test-R2']   = round(r2_score(Test_Y, test_predictions),4)
    model_evaluation.loc['Multiple Linear Regression','Train-RSS'] = round(np.sum(np.square(Train_Y-train_predictions)),4)
    model_evaluation.loc['Multiple Linear Regression','Test-RSS']  = round(np.sum(np.square(Test_Y-test_predictions)),4)
    model_evaluation.loc['Multiple Linear Regression','Train-MSE'] = round(mean_squared_error(Train_Y, train_predictions),4)
    model_evaluation.loc['Multiple Linear Regression','Test-MSE']  = round(mean_squared_error(Test_Y, test_predictions),4)
    model_evaluation.loc['Multiple Linear Regression','Train-RMSE']= round(np.sqrt(mean_squared_error(Train_Y, train_predictions)),4)
    model_evaluation.loc['Multiple Linear Regression','Test-RMSE'] = round(np.sqrt(mean_squared_error(Test_Y, test_predictions)),4)

    # Plotting y_test and y_pred to understand the spread.
    plt.figure(figsize=[15,4])

    plt.subplot(1,2,1)
    sns.distplot((Train_Y - train_predictions))
    plt.title('Error Terms')          
    plt.xlabel('Errors') 

    plt.subplot(1,2,2)
    plt.scatter(Train_Y,train_predictions)
    plt.plot([Train_Y.min(),Train_Y.max()],[Train_Y.min(),Train_Y.max()], 'r--')
    plt.title('Train vs Prediction')         
    plt.xlabel('y_train')                       
    plt.ylabel('y_pred')                       
    plt.show()

---

##  Multiple Linear Regression (MLR)

<img src="https://raw.githubusercontent.com/Masterx-AI/Project_BoomBikes_Share_Prediction/main/mr.png" style="width: 600px;float: left;"/>

In [ ]:
print('<<<-----------------------------------\033[1m Evaluating Multiple Linear Regression Model \033[0m----------------------------------->>>')

# Lấy các features được RFE chọn
selected_cols = list(Train_X_std.loc[:, rfe.support_].columns)
print('Training with RFE-selected features:', selected_cols)

MLR = LinearRegression()
MLR.fit(Train_X_std[selected_cols], Train_Y)

train_predictions = MLR.predict(Train_X_std[selected_cols])
test_predictions  = MLR.predict(Test_X_std[selected_cols])

print('\nThe Coefficient of the Regression Model was found to be ', MLR.coef_)
print('The Intercept of the Regression Model was found to be ', MLR.intercept_)

# Lưu coefficients
coef_df = pd.DataFrame({
    'Feature': ['Intercept'] + selected_cols,
    'Coefficient': [MLR.intercept_] + list(MLR.coef_)
})
coef_df.to_csv(TABLES_DIR / '06_model_coefficients.csv', index=False)
print('\nModel coefficients saved.')

# Lưu test predictions
pred_df = pd.DataFrame({'Actual': Test_Y.values, 'Predicted': test_predictions})
pred_df.to_csv(TABLES_DIR / '08_test_predictions.csv', index=False)
print('Test predictions saved.')

evaluate_mlr(train_predictions, test_predictions)


In [ ]:
#Model Evaluation Summary

print('\n\033[1mModel Evaluation Results:\033[0m')
display(model_evaluation)

---

# <center> 7. Project Outcomes & Conclusion

## Key Outcomes

* The original dataset contains **545 housing records**, with **12 input variables** and one target variable, `price`. After removing area outliers using the IQR method, **533 records** remained for model development.

* Exploratory Data Analysis showed that house price is **right-skewed**, with most observations concentrated in the lower-to-middle price range and several high-price observations forming a long right tail.

* Several housing attributes showed noticeable associations with price. In particular, larger area, more bathrooms, air conditioning, additional stories, parking availability, and location in a preferred area were generally associated with higher house prices.

* Categorical variables, including `mainroad`, `guestroom`, `basement`, `hotwaterheating`, `airconditioning`, `prefarea`, and `furnishingstatus`, were converted into numerical variables using one-hot encoding.

* The processed dataset was divided into training and testing sets using an 80:20 ratio. Feature scaling was fitted only on the training set, and the same fitted scaler was then applied to the testing set to avoid data leakage during standardization.

* Multicollinearity was examined using the Variance Inflation Factor. All calculated VIF values were below 2, indicating that no serious multicollinearity was detected among the encoded input variables.

* Recursive Feature Elimination was applied with `LinearRegression` as the estimator. RFE selected 10 out of the 13 encoded input features and removed `bedrooms`, `guestroom_yes`, and `furnishingstatus_semi-furnished`.

* A Multiple Linear Regression model was trained using the 10 RFE-selected features. The model achieved a training R² of approximately **0.658** and a testing R² of approximately **0.722**.

* The testing RMSE was approximately **981,706 price units**, meaning that the model's predictions differed from the actual house prices by about 0.98 million price units on average when larger errors were given more weight.

* The testing performance was slightly better than the training performance on the selected train-test split. Therefore, no obvious sign of overfitting was observed in this split. However, cross-validation would be required to determine whether this result remains stable across different data partitions.

* RFE reduced the number of input features while maintaining almost the same predictive performance as the model using all encoded features. Its main benefit in this project was therefore model simplification rather than a substantial improvement in accuracy.

* Residual analysis showed that the model performed more consistently for houses in the middle price range, while larger prediction errors tended to occur for some high-price properties.

* Overall, Multiple Linear Regression provides an interpretable baseline model for the housing-price prediction task, but its linear structure may not fully represent more complex relationships in the data.

## Limitations

* The dataset is relatively small, with only 545 original observations and 533 observations remaining after area-outlier filtering. Consequently, model performance may vary depending on how the data is divided into training and testing sets.

* The model was evaluated using a single train-test split. Therefore, the reported metrics may not fully represent the model's average performance on other unseen samples.

* The number of features selected by RFE was manually set to 10. The project did not use cross-validation to determine whether 10 was the optimal number of features.

* The IQR thresholds used for area-outlier removal were calculated before the train-test split. A stricter machine-learning workflow would estimate preprocessing rules using the training data only.

* Multiple Linear Regression assumes that the relationship between the predictors and house price can be represented approximately by a linear equation. Nonlinear relationships and interactions between housing attributes may therefore be missed.

* Although VIF results did not indicate serious multicollinearity, other regression assumptions, such as constant residual variance, residual normality, and the influence of high-leverage observations, were not examined comprehensively.

* Removing area outliers may reduce the influence of extreme observations, but some of the removed properties may represent valid large houses rather than data errors.

* The dataset does not provide detailed contextual variables such as exact geographic location, neighborhood characteristics, construction year, property condition, or market conditions. These omitted variables may account for a substantial proportion of unexplained price variation.

* The monetary unit of the `price` variable is not explicitly documented in the dataset. Therefore, evaluation results should be reported in “price units” unless the original data source confirms a specific currency.

## Future Improvements

* Apply k-fold cross-validation to obtain more stable estimates of R², MAE, MSE, and RMSE across multiple data partitions.

* Use `RFECV` or another cross-validation-based feature-selection method to determine the most appropriate number of input features instead of fixing the number at 10.

* Build a complete preprocessing pipeline so that encoding, scaling, feature selection, and model training are performed consistently and can be reproduced without manual intermediate steps.

* Calculate outlier thresholds using only the training data, or compare model performance with and without outlier removal before deciding whether extreme observations should be excluded.

* Add baseline models, such as predicting the mean house price or using a regression model based only on area, to provide a clearer reference for evaluating model improvement.

* Perform additional residual diagnostics, including residual-versus-fitted plots, Q–Q plots, heteroscedasticity tests, and influential-point analysis.

* Explore transformations such as the logarithm of `price` or `area` to reduce skewness and potentially improve the linear relationship between the predictors and the target.

* Perform systematic feature engineering, including interaction terms, polynomial terms, area-per-bedroom variables, and combined indicators for housing facilities.

* Compare Multiple Linear Regression with regularized linear models such as Ridge, Lasso, and Elastic Net.

* Evaluate nonlinear models such as Random Forest, Gradient Boosting, and XGBoost to determine whether they can better capture complex patterns in housing prices.

* Collect or incorporate additional explanatory variables, especially location, neighborhood quality, house age, property condition, and market-related information.

* Report both predictive performance and model interpretability when comparing alternative algorithms, since more complex models may improve accuracy while being more difficult to explain.
